In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.utils.db import get_engine

engine = get_engine()

In [2]:
point_df = pd.read_sql(
    "SELECT * FROM accounts_pointhistory",
    engine
)

print(point_df.shape)
point_df.head()

(2338918, 5)


,id,delta_point,created_at,user_id,user_question_record_id
0,790629,9,2023-04-28 12:27:49,849436,771777.0
1,790652,9,2023-04-28 12:28:02,849436,771800.0
2,790664,5,2023-04-28 12:28:09,849436,771812.0
3,790680,13,2023-04-28 12:28:16,849436,771828.0
4,790703,5,2023-04-28 12:28:26,849436,771851.0


In [3]:
point_df.info()

print(point_df.isnull().sum())

print(point_df['created_at'].min())
print(point_df['created_at'].max())

print(point_df['delta_point'].describe())

<class 'pandas.DataFrame'>
RangeIndex: 2338918 entries, 0 to 2338917
Data columns (total 5 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   id                       int64         
 1   delta_point              int64         
 2   created_at               datetime64[us]
 3   user_id                  int64         
 4   user_question_record_id  float64       
dtypes: datetime64[us](1), float64(1), int64(3)
memory usage: 89.2 MB
id                            0
delta_point                   0
created_at                    0
user_id                       0
user_question_record_id    2992
dtype: int64
2023-04-28 12:27:49
2024-05-08 01:36:18
count    2.338918e+06
mean     8.610862e-01
std      5.885849e+01
min     -1.000000e+03
25%      7.000000e+00
50%      1.000000e+01
75%      1.300000e+01
max      1.000000e+03
Name: delta_point, dtype: float64


- 전체 2,338,918행
- 기간: 2023-04-28 12:27:49 ~ 2024-05-08 01:36:18
- id, delta_point, created_at, user_id 결측 없음
- user_question_record_id만 2,992건 결측
- delta_point: -1000 ~ +1000
- 음수값이 존재 -> delta_point < 0을 재화 소모 후보로 볼 수 있음
- user_question_record_id 결측은 일단 삭제 안함. -> 질문과 관련 없는 포인트 증감 기록일 수도 있음.

In [5]:
point_preprocessed = point_df.copy()

In [8]:
# ID 컬럼 타입 통일
point_preprocessed['id'] = (
    point_preprocessed['id']
    .astype('Int64')
    .astype('string')
)

point_preprocessed['user_id'] = (
    point_preprocessed['user_id']
    .astype('Int64')
    .astype('string')
)

In [9]:
# 결측이 있으므로 nullable Int64 → string
point_preprocessed['user_question_record_id'] = (
    point_preprocessed['user_question_record_id']
    .astype('Int64')
    .astype('string')
)

In [10]:
# 날짜형
point_preprocessed['created_at'] = pd.to_datetime(
    point_preprocessed['created_at'],
    errors='coerce'
)

In [11]:
# 중복 확인
print("전체 중복:", point_preprocessed.duplicated().sum())
print("id 중복:", point_preprocessed['id'].duplicated().sum())

전체 중복: 0
id 중복: 0


In [12]:
# 최종 확인
print(point_preprocessed.info())

<class 'pandas.DataFrame'>
RangeIndex: 2338918 entries, 0 to 2338917
Data columns (total 5 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   id                       string        
 1   delta_point              int64         
 2   created_at               datetime64[us]
 3   user_id                  string        
 4   user_question_record_id  string        
dtypes: datetime64[us](1), int64(1), string(3)
memory usage: 89.2 MB
None


In [13]:
output_path = (
    PROJECT_ROOT
    / 'data'
    / 'interim'
    / 'accounts_pointhistory_preprocessed.csv'
)

point_preprocessed.to_csv(
    output_path,
    index=False
)

print("저장 완료:", output_path)
print("shape:", point_preprocessed.shape)

저장 완료: /Users/jsun99/Workspace/02_develop/projects/team4-final-project/data/interim/accounts_pointhistory_preprocessed.csv
shape: (2338918, 5)
